# Persiapan Data Labeling — Pisah Overlap
**Masalah:** 13.210 data berlabel manual ada di dalam `data_cleaned.csv`  
**Solusi:** Pisahkan sebelum labeling otomatis  

| Kelompok | Jumlah | Perlakuan |
|---|---|---|
| Sudah berlabel (overlap di cleaned) | ~9.922 | Ambil label dari data_labeling.csv |
| Berlabel tapi tidak di cleaned | ~5.324 | Tambahkan langsung |
| Belum berlabel (cleaned - overlap) | ~53.622 | IndoBERT labeli otomatis |
| **Total akhir** | **~68.868** | **labelled_data_final.csv** |

## Cell 1 — Setup & Load

In [2]:
import pandas as pd
import numpy as np
import os

BASE  = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling'

FILE_LABELED  = os.path.join(BASE, 'data_labeling_cleaned.csv')       # 18.534 label manual
FILE_CLEANED  = os.path.join(BASE, 'data_cleaning', 'data_cleaned.csv')  # 63.544 hasil cleaning
FILE_UNLABELED_OUT = os.path.join(BASE, 'data_for_auto_labeling.csv')    # output: belum berlabel
FILE_LABELED_OUT   = os.path.join(BASE, 'data_already_labeled.csv')      # output: sudah berlabel
FILE_FINAL         = os.path.join(BASE, 'labelled_data_final.csv')       # output akhir

df_label = pd.read_csv(FILE_LABELED)
df_clean = pd.read_csv(FILE_CLEANED)

# Normalisasi text untuk matching
df_label['text_norm'] = df_label['text'].str.strip().str.lower()
df_clean['text_norm'] = df_clean['text'].str.strip().str.lower()

print(f'Data berlabel (Seprianto) : {len(df_label):,}')
print(f'Data cleaned              : {len(df_clean):,}')

print(f'\nDistribusi label manual:')
for lbl, cnt in df_label['label_pks'].value_counts().items():
    pct = cnt/len(df_label)*100
    bar = chr(9608) * int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:5.1f}%) {bar}')

Data berlabel (Seprianto) : 13,552
Data cleaned              : 63,538

Distribusi label manual:
  keluhan   :  9,497 ( 70.1%) ███████████████████████
  saran     :  2,622 ( 19.3%) ██████
  pujian    :  1,433 ( 10.6%) ███


## Cell 2 — Identifikasi 3 Kelompok Data

In [3]:
SEP = '=' * 55

# ── Kelompok A: Ada di cleaned DAN sudah berlabel ────────────────────────────
labeled_texts = set(df_label['text_norm'].dropna())

mask_overlap  = df_clean['text_norm'].isin(labeled_texts)
df_overlap    = df_clean[mask_overlap].copy()   # di cleaned, sudah berlabel
df_no_label   = df_clean[~mask_overlap].copy()  # di cleaned, belum berlabel

# ── Kelompok B: Ada di labeled tapi TIDAK ada di cleaned ─────────────────────
clean_texts  = set(df_clean['text_norm'].dropna())
df_label_only = df_label[~df_label['text_norm'].isin(clean_texts)].copy()

print(SEP)
print('IDENTIFIKASI 3 KELOMPOK DATA')
print(SEP)
print(f'\nKelompok A — Di cleaned + sudah berlabel (overlap):')
print(f'  Jumlah : {len(df_overlap):,} baris')
print(f'  → Akan di-merge dengan label dari data_labeling.csv')

print(f'\nKelompok B — Di labeled tapi tidak di cleaned:')
print(f'  Jumlah : {len(df_label_only):,} baris')
print(f'  → Akan ditambahkan langsung ke dataset final')

print(f'\nKelompok C — Di cleaned, belum berlabel sama sekali:')
print(f'  Jumlah : {len(df_no_label):,} baris')
print(f'  → Akan dilabeli otomatis oleh IndoBERT')

print(f'\nTotal akhir estimasi: {len(df_overlap)+len(df_label_only)+len(df_no_label):,} baris')

IDENTIFIKASI 3 KELOMPOK DATA

Kelompok A — Di cleaned + sudah berlabel (overlap):
  Jumlah : 9,006 baris
  → Akan di-merge dengan label dari data_labeling.csv

Kelompok B — Di labeled tapi tidak di cleaned:
  Jumlah : 4,590 baris
  → Akan ditambahkan langsung ke dataset final

Kelompok C — Di cleaned, belum berlabel sama sekali:
  Jumlah : 54,532 baris
  → Akan dilabeli otomatis oleh IndoBERT

Total akhir estimasi: 68,128 baris


## Cell 3 — Proses Kelompok A: Merge Label Manual ke Data Cleaned

In [4]:
# Merge label dari df_label ke df_overlap berdasarkan text
label_map = df_label.set_index('text_norm')['label_pks'].to_dict()

df_overlap['label_pks'] = df_overlap['text_norm'].map(label_map)
df_overlap['label_enc'] = df_overlap['label_pks'].map({'keluhan':0,'saran':1,'pujian':2})
df_overlap['confidence']= 1.0   # label manual = kepercayaan penuh
df_overlap['source']    = 'manual'

# Cek apakah ada yang gagal di-map (seharusnya 0)
unmapped = df_overlap['label_pks'].isna().sum()
print(f'Kelompok A — hasil merge:')
print(f'  Total     : {len(df_overlap):,}')
print(f'  Berhasil  : {df_overlap["label_pks"].notna().sum():,}')
print(f'  Gagal map : {unmapped:,}')

if unmapped > 0:
    print(f'  Contoh yang gagal:')
    print(df_overlap[df_overlap['label_pks'].isna()][['text']].head(3).to_string())

print(f'\nDistribusi label Kelompok A:')
for lbl, cnt in df_overlap['label_pks'].value_counts().items():
    pct = cnt/len(df_overlap)*100
    print(f'  {lbl:10s}: {cnt:6,} ({pct:5.1f}%)')

Kelompok A — hasil merge:
  Total     : 9,006
  Berhasil  : 9,006
  Gagal map : 0

Distribusi label Kelompok A:
  keluhan   :  5,972 ( 66.3%)
  saran     :  1,788 ( 19.9%)
  pujian    :  1,246 ( 13.8%)


## Cell 4 — Proses Kelompok B: Data Berlabel yang Tidak Ada di Cleaned

In [5]:
# Kelompok B: data dari Seprianto yang tidak masuk combined_all_data
# Tambahkan langsung — label sudah ada, tinggal sesuaikan format

df_label_only = df_label_only.copy()
df_label_only['label_enc']  = df_label_only['label_pks'].map({'keluhan':0,'saran':1,'pujian':2})
df_label_only['confidence'] = 1.0
df_label_only['source']     = 'manual'
df_label_only['platform']   = 'Instagram'  # semua dari scraping Instagram lama

print(f'Kelompok B — data berlabel yang tidak ada di cleaned:')
print(f'  Total   : {len(df_label_only):,}')
print(f'\nDistribusi label Kelompok B:')
for lbl, cnt in df_label_only['label_pks'].value_counts().items():
    pct = cnt/len(df_label_only)*100
    print(f'  {lbl:10s}: {cnt:6,} ({pct:5.1f}%)')

print(f'\nContoh teks Kelompok B:')
for _, row in df_label_only.sample(3, random_state=42).iterrows():
    print(f'  [{row.label_pks}] {str(row.text)[:80]}')

Kelompok B — data berlabel yang tidak ada di cleaned:
  Total   : 9,431

Distribusi label Kelompok B:
  keluhan   :  4,559 ( 48.3%)
  pujian    :  3,874 ( 41.1%)
  saran     :    998 ( 10.6%)

Contoh teks Kelompok B:
  [saran] Hallo min, bayi saya mau imunisasi di puskesmas tpi faskes bpjsnya bukan di pusk
  [saran] Min.. aku udeh nunggak dr min.. klo mao aktifin lagi.. dan dipindah ke bpjs grat
  [keluhan] Salah pak-ibu selamat pagi peraturan ijin Dan lain-lain emenkeu tahun hilang Pak


## Cell 5 — Simpan Kelompok C untuk Auto-Labeling IndoBERT

In [6]:
# Kelompok C: simpan sebagai file input untuk notebook auto-labeling
cols_save = ['id','timestamp','ownerUsername','text',
             'likesCount','postUrl','commentUrl','platform','source_file']

# Sesuaikan kolom yang ada
cols_available = [c for c in cols_save if c in df_no_label.columns]
df_no_label[cols_available].to_csv(FILE_UNLABELED_OUT, index=False, encoding='utf-8-sig')

print(f'Kelompok C tersimpan: {FILE_UNLABELED_OUT}')
print(f'  Total : {len(df_no_label):,} baris — siap dilabeli IndoBERT')

# Simpan Kelompok A sebagai referensi
cols_labeled = [c for c in cols_available + ['label_pks','label_enc','confidence','source'] 
                if c in df_overlap.columns]
df_overlap[cols_labeled].to_csv(FILE_LABELED_OUT, index=False, encoding='utf-8-sig')
print(f'Kelompok A tersimpan: {FILE_LABELED_OUT}')
print(f'  Total : {len(df_overlap):,} baris berlabel manual')

print(f'\nInstruksi selanjutnya:')
print(f'  1. Buka Labeling_Otomatis_IndoBERT.ipynb')
print(f'  2. Atur FILE_LABELED   = data_labeling.csv       (18.534 seed)')
print(f'  3. Atur FILE_UNLABELED = data_for_auto_labeling.csv ({len(df_no_label):,} baris)')
print(f'  4. Jalankan Cell 1 sampai Cell 8')
print(f'  5. Setelah selesai, jalankan Cell 6 di notebook ini untuk gabungkan')

Kelompok C tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling\data_for_auto_labeling.csv
  Total : 54,532 baris — siap dilabeli IndoBERT
Kelompok A tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling\data_already_labeled.csv
  Total : 9,006 baris berlabel manual

Instruksi selanjutnya:
  1. Buka Labeling_Otomatis_IndoBERT.ipynb
  2. Atur FILE_LABELED   = data_labeling.csv       (18.534 seed)
  3. Atur FILE_UNLABELED = data_for_auto_labeling.csv (54,532 baris)
  4. Jalankan Cell 1 sampai Cell 8
  5. Setelah selesai, jalankan Cell 6 di notebook ini untuk gabungkan


In [8]:
df_overlap.head()

,id,timestamp,ownerUsername,text,likesCount,postUrl,commentUrl,platform,source_file,text_norm,label_pks,label_enc,confidence,source
243,18088145777029061,2026-05-11 16:17:43+00:00,prabowo.da,Lahan basah lahan basah,0,https://www.instagram.com/reel/DXd-RSckWCU/,https://www.instagram.com/reel/DXd-RSckWCU/?co...,Instagram,dataset_instagram_20260525_132650.csv,lahan basah lahan basah,keluhan,0,1.0,manual
5779,17899342884303621,2025-11-02 12:23:10+00:00,ibrohom28,Taxpayer Identity validation failed.,0,https://www.instagram.com/p/DNCnMRdTWgC/,https://www.instagram.com/p/DNCnMRdTWgC/?comme...,Instagram,dataset_instagram_20260525_140226.csv,taxpayer identity validation failed.,keluhan,0,1.0,manual
6373,17892547551334234,2026-04-30 00:58:29+00:00,andryacp,Terima kasih infonya,1,https://www.instagram.com/reel/DXrT9W6oHrv/,https://www.instagram.com/reel/DXrT9W6oHrv/?co...,Instagram,dataset_instagram_20260525_140900.csv,terima kasih infonya,pujian,2,1.0,manual
6374,18088135541611507,2026-05-06 13:53:00+00:00,herukustaman_,Cek DM min,0,https://www.instagram.com/reel/DXrT9W6oHrv/,https://www.instagram.com/reel/DXrT9W6oHrv/?co...,Instagram,dataset_instagram_20260525_140900.csv,cek dm min,saran,1,1.0,manual
6440,17899105932431853,2026-04-28 23:26:49+00:00,tantyfaridmultianty,Terima Kasih informasinya,3,https://www.instagram.com/reel/DXrT9W6oHrv/,https://www.instagram.com/reel/DXrT9W6oHrv/?co...,Instagram,dataset_instagram_20260525_140900.csv,terima kasih informasinya,pujian,2,1.0,manual


## Cell 6 — Gabungkan Semua Setelah Auto-Labeling Selesai
> Jalankan cell ini SETELAH notebook `Labeling_Otomatis_IndoBERT.ipynb` selesai dijalankan
> dan menghasilkan file `labelled_data_final.csv` dari Kelompok C

In [11]:
# ══════════════════════════════════════════════════════════════
# CELL 6 FIX — Gabungkan Semua dengan Benar
# Perbaikan:
#   1. Deduplikasi sebelum gabung → cegah baris double
#   2. Pertahankan semua kolom penting
#   3. Label manual prioritas atas label otomatis
# ══════════════════════════════════════════════════════════════

import pandas as pd
import os

BASE = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling'

# ── Load semua file ───────────────────────────────────────────────────────────
FILE_AUTO  = os.path.join(BASE, 'labelled_data_final.csv')      # output IndoBERT
FILE_A     = os.path.join(BASE, 'data_already_labeled.csv')     # Kelompok A
FILE_FINAL = os.path.join(BASE, 'labelled_data_final_COMPLETE.csv')

df_auto = pd.read_csv(FILE_AUTO)    # hasil IndoBERT (Kelompok C)
df_A    = pd.read_csv(FILE_A)       # Kelompok A (overlap, label manual)
df_B    = df_label_only.copy()      # Kelompok B (label manual, tidak di cleaned)

print('='*55)
print('DATA SEBELUM DEDUPLIKASI')
print('='*55)
print(f'  df_auto (IndoBERT)  : {len(df_auto):,}')
print(f'  df_A (manual)       : {len(df_A):,}')
print(f'  df_B (manual)       : {len(df_B):,}')
print(f'  Total mentah        : {len(df_auto)+len(df_A)+len(df_B):,}')

# ── Normalisasi text untuk deduplikasi ───────────────────────────────────────
for df in [df_auto, df_A, df_B]:
    df['text_norm'] = df['text'].str.strip().str.lower()

# ── Hapus dari df_auto semua yang sudah ada di Kelompok A ────────────────────
# Kelompok A = data yang sudah berlabel manual dan ada di cleaned
# IndoBERT juga melabeli ini → harus dibuang dari df_auto
labeled_manual_texts = set(df_A['text_norm']) | set(df_B['text_norm'])
df_C = df_auto[~df_auto['text_norm'].isin(labeled_manual_texts)].copy()

print(f'\n  df_auto setelah hapus overlap: {len(df_C):,}')
print(f'  (dihapus {len(df_auto)-len(df_C):,} duplikat dengan data manual)')

# ── Siapkan kolom seragam ────────────────────────────────────────────────────
COLS_FINAL = [
    'id', 'timestamp', 'ownerUsername', 'text',
    'likesCount', 'postUrl', 'commentUrl',
    'platform', 'source_file',
    'label_pks', 'label_enc', 'confidence', 'source'
]

def prep_df(df, source_name):
    df = df.copy()

    # Label encoding
    if 'label_enc' not in df.columns:
        df['label_enc'] = df['label_pks'].map({'keluhan':0,'saran':1,'pujian':2})

    # Confidence
    if 'confidence' not in df.columns:
        df['confidence'] = 1.0 if source_name == 'manual' else df.get('confidence', 0.95)

    # Source
    df['source'] = source_name

    # Pastikan semua kolom ada (isi NaN jika tidak ada)
    for col in COLS_FINAL:
        if col not in df.columns:
            df[col] = None

    return df[COLS_FINAL]

df_A_prep = prep_df(df_A, 'manual')
df_B_prep = prep_df(df_B, 'manual')
df_C_prep = prep_df(df_C, 'auto')

# ── Gabungkan ─────────────────────────────────────────────────────────────────
df_final = pd.concat([df_A_prep, df_B_prep, df_C_prep], ignore_index=True)

# Deduplikasi akhir berdasarkan text — label manual prioritas (sudah di atas)
df_final['_text_norm'] = df_final['text'].str.strip().str.lower()
df_final = df_final.drop_duplicates(subset=['_text_norm'], keep='first')
df_final = df_final.drop(columns=['_text_norm'])

# Acak urutan
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# ── Simpan ────────────────────────────────────────────────────────────────────
df_final.to_csv(FILE_FINAL, index=False, encoding='utf-8-sig')

# ── Laporan ───────────────────────────────────────────────────────────────────
SEP = '=' * 55
print(f'\n{SEP}')
print('DATASET BERLABEL FINAL — RINGKASAN')
print(SEP)
print(f'  Kelompok A (manual, overlap)  : {len(df_A_prep):,}')
print(f'  Kelompok B (manual, tambahan) : {len(df_B_prep):,}')
print(f'  Kelompok C (auto IndoBERT)    : {len(df_C_prep):,}')
print(f'  Total sebelum dedup akhir     : {len(df_A_prep)+len(df_B_prep)+len(df_C_prep):,}')
print(f'  Total setelah dedup akhir     : {len(df_final):,}')

print(f'\nDistribusi label final:')
for lbl, cnt in df_final['label_pks'].value_counts().items():
    pct = cnt/len(df_final)*100
    bar = chr(9608) * int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%) {bar}')

print(f'\nDistribusi sumber:')
for src, cnt in df_final['source'].value_counts().items():
    pct = cnt/len(df_final)*100
    print(f'  {src:8s}: {cnt:6,} ({pct:.1f}%)')

print(f'\nKolom yang tersimpan:')
print(f'  {list(df_final.columns)}')

print(f'\nFile tersimpan: {FILE_FINAL}')
print(f'Total baris   : {len(df_final):,}')

DATA SEBELUM DEDUPLIKASI
  df_auto (IndoBERT)  : 73,066
  df_A (manual)       : 9,006
  df_B (manual)       : 9,431
  Total mentah        : 91,503

  df_auto setelah hapus overlap: 54,532
  (dihapus 18,534 duplikat dengan data manual)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13656\3784838917.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_A_prep, df_B_prep, df_C_prep], ignore_index=True)



DATASET BERLABEL FINAL — RINGKASAN
  Kelompok A (manual, overlap)  : 9,006
  Kelompok B (manual, tambahan) : 9,431
  Kelompok C (auto IndoBERT)    : 54,532
  Total sebelum dedup akhir     : 72,969
  Total setelah dedup akhir     : 69,184

Distribusi label final:
  keluhan   : 47,779 (69.1%) ███████████████████████
  saran     : 13,651 (19.7%) ██████
  pujian    :  7,754 (11.2%) ███

Distribusi sumber:
  auto    : 54,206 (78.4%)
  manual  : 14,978 (21.6%)

Kolom yang tersimpan:
  ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 'postUrl', 'commentUrl', 'platform', 'source_file', 'label_pks', 'label_enc', 'confidence', 'source']

File tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling\labelled_data_final_COMPLETE.csv
Total baris   : 69,184


Cek data Overlapping

In [4]:
import pandas as pd
import os

BASE_DIR = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_method\data_labelling'

df_semua = pd.read_csv(os.path.join(BASE_DIR,'data_cleaning',  'data_cleaned.csv'))
df_lama  = pd.read_csv(os.path.join(BASE_DIR, 'data_labeling_cleaned.csv'))

print(f"Total data cleaned : {len(df_semua)}")
print(f"Total data lama    : {len(df_lama)}")

# --- Cek kolom yang sama di kedua file ---
print(f"\nKolom cleaned_data : {df_semua.columns.tolist()}")
print(f"Kolom data lama    : {df_lama.columns.tolist()}")

# --- Gunakan commentUrl sebagai key unik ---
# commentUrl bersifat unik per komentar dan tidak kehilangan presisi
key_col = 'commentUrl'

# Cek apakah commentUrl ada di kedua file
print(f"\nSampel commentUrl cleaned_data:")
print(df_semua[key_col].head(3).tolist())
print(f"\nSampel commentUrl data lama:")
print(df_lama[key_col].head(3).tolist())

# --- FILTER DATA BARU ---
url_lama   = set(df_lama[key_col].astype(str).str.strip())
df_baru    = df_semua[~df_semua[key_col].astype(str).str.strip().isin(url_lama)]
df_overlap = df_semua[df_semua[key_col].astype(str).str.strip().isin(url_lama)]

print(f"\nData overlap (sudah berlabel) : {len(df_overlap)} baris")
print(f"Data baru (belum berlabel)    : {len(df_baru)} baris")

# --- SIMPAN ---
output_path = os.path.join(BASE_DIR, 'cleaned_data_baru.csv')
df_baru.to_csv(output_path, index=False)
print(f"\n✅ Data baru tersimpan: {output_path}")

Total data cleaned : 63538
Total data lama    : 13552

Kolom cleaned_data : ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 'postUrl', 'commentUrl', 'platform', 'source_file']
Kolom data lama    : ['id', 'timestamp', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'ownerUsername', 'text', 'label_pks']

Sampel commentUrl cleaned_data:
['https://www.instagram.com/reel/DXd-RSckWCU/?comment_id=18131249071561283', 'https://www.instagram.com/reel/DXd-RSckWCU/?comment_id=18049919705732880', 'https://www.instagram.com/reel/DXd-RSckWCU/?comment_id=18191321884373494']

Sampel commentUrl data lama:
['https://www.instagram.com/ditjenpajakri/p/DPQjaH5ktZj/?comment_id=18080384687097420', 'https://www.instagram.com/ditjenpajakri/p/DPQjaH5ktZj/?comment_id=17946622826916922', 'https://www.instagram.com/ditjenpajakri/p/DPQjaH5ktZj/?comment_id=17980771046917323']

Data overlap (sudah berlabel) : 13536 baris
Data baru (belum berlabel)    : 50002 baris

✅ Data baru tersimpan: C:\Users\Leno